# 01 — Raw Data Audit

**Project:** Orbital Congestion in Low Earth Orbit

This notebook is the first analytical step of the project.

Its purpose is **not** to create the final visualizations. It is meant to answer:

> What data do we actually have, how compatible are the sources, and can they support the project we proposed?

The audit focuses on:
- file discovery;
- schema inspection;
- candidate object identifiers;
- missingness;
- duplicated records;
- temporal variables;
- object-type / status variables;
- orbital altitude variables;
- feasibility of reconstructing the historical in-orbit population.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data.audit import discover_files, load_table, audit_dataframe, compact_summary

RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR


## 1. Discover raw files

Place your downloaded sources in:

- `data/raw/ucs/`
- `data/raw/celestrak/`
- `data/raw/esa/`

The next cell lists supported tabular files.


In [ ]:
source_dirs = {
    "ucs": RAW_DIR / "ucs",
    "celestrak": RAW_DIR / "celestrak",
    "esa": RAW_DIR / "esa",
}

discovered = {
    source: discover_files(path)
    for source, path in source_dirs.items()
}

for source, files in discovered.items():
    print(f"\n{source.upper()} ({len(files)} files)")
    for f in files:
        print(" -", f.relative_to(PROJECT_ROOT))


## 2. Load one candidate file per source

The notebook deliberately does not guess which file is correct if you place multiple versions in the same folder.

Set the paths below once the raw data has been added.


In [ ]:
# Example:
# UCS_FILE = PROJECT_ROOT / "data/raw/ucs/ucs-satellite-database.csv"
# CELESTRAK_FILE = PROJECT_ROOT / "data/raw/celestrak/satcat.csv"

UCS_FILE = None
CELESTRAK_FILE = None

ucs = load_table(UCS_FILE) if UCS_FILE else None
celestrak = load_table(CELESTRAK_FILE) if CELESTRAK_FILE else None

print("UCS loaded:", None if ucs is None else ucs.shape)
print("CelesTrak loaded:", None if celestrak is None else celestrak.shape)


## 3. Basic schema audit


In [ ]:
def show_audit(df, name):
    if df is None:
        print(f"{name}: not loaded")
        return None
    audit = audit_dataframe(df, name)
    display(compact_summary(audit))
    return audit

ucs_audit = show_audit(ucs, "UCS")
celestrak_audit = show_audit(celestrak, "CelesTrak / SATCAT")


## 4. Inspect columns and sample records

Pay particular attention to:
- a stable catalogue ID, preferably NORAD CAT ID if available;
- launch and decay dates;
- object type / operational status;
- country and operator;
- perigee, apogee, or another reliable altitude measure.


In [ ]:
if ucs is not None:
    print("UCS columns")
    display(pd.DataFrame({"column": ucs.columns.astype(str)}))
    display(ucs.head())

if celestrak is not None:
    print("CelesTrak columns")
    display(pd.DataFrame({"column": celestrak.columns.astype(str)}))
    display(celestrak.head())


## 5. Missingness

We are especially interested in missingness in the variables required by the project.

A field can exist and still be unusable if it is mostly missing or inconsistently populated.


In [ ]:
def missingness_table(df):
    return (
        df.isna()
        .mean()
        .mul(100)
        .sort_values(ascending=False)
        .rename("missing_pct")
        .to_frame()
    )

if ucs is not None:
    print("UCS missingness")
    display(missingness_table(ucs).head(30))

if celestrak is not None:
    print("CelesTrak missingness")
    display(missingness_table(celestrak).head(30))


## 6. Candidate join key

This is one of the most important checks in the project.

We need to determine whether the active-satellite source can be linked reliably to the wider catalogue.

Once the real column names are known, set them below and evaluate:
- uniqueness within each dataset;
- missingness;
- overlap between sources;
- unmatched records.


In [ ]:
UCS_ID_COL = None
CELESTRAK_ID_COL = None

if ucs is not None and celestrak is not None and UCS_ID_COL and CELESTRAK_ID_COL:
    u_ids = pd.to_numeric(ucs[UCS_ID_COL], errors="coerce").dropna().astype("int64")
    c_ids = pd.to_numeric(celestrak[CELESTRAK_ID_COL], errors="coerce").dropna().astype("int64")

    print("UCS unique IDs:", u_ids.nunique(), "/", len(u_ids))
    print("CelesTrak unique IDs:", c_ids.nunique(), "/", len(c_ids))

    overlap = set(u_ids) & set(c_ids)
    print("ID overlap:", len(overlap))
    print("Share of UCS IDs found in CelesTrak:",
          round(len(overlap) / u_ids.nunique() * 100, 2), "%")
else:
    print("Set UCS_ID_COL and CELESTRAK_ID_COL after inspecting the real schemas.")


## 7. Temporal feasibility

The project distinguishes:
- cumulative historical launches;
- objects actually remaining in orbit.

The stronger historical analysis requires launch dates and, where applicable, decay dates.

For an object *i*, a simple reconstructed in-orbit interval is conceptually:

`launch_date <= t < decay_date`

or, if no decay has occurred:

`launch_date <= t`

This can only be used after checking the source semantics carefully.


In [ ]:
# Fill these only after schema inspection.
LAUNCH_DATE_COL = None
DECAY_DATE_COL = None

if celestrak is not None and LAUNCH_DATE_COL:
    launch = pd.to_datetime(celestrak[LAUNCH_DATE_COL], errors="coerce")
    print("Launch date coverage:")
    print("min:", launch.min())
    print("max:", launch.max())
    print("missing:", launch.isna().mean().round(4))

if celestrak is not None and DECAY_DATE_COL:
    decay = pd.to_datetime(celestrak[DECAY_DATE_COL], errors="coerce")
    print("\nDecay date coverage:")
    print("min:", decay.min())
    print("max:", decay.max())
    print("missing:", decay.isna().mean().round(4))


## 8. Orbital altitude feasibility

The proposal focuses on congestion within LEO and on heavily used altitude bands.

Preferred inputs are perigee and apogee, from which we can derive a simple mean altitude:

`mean_altitude_km = (perigee_km + apogee_km) / 2`

This is an analytical convenience, not a full description of the orbit.


In [ ]:
PERIGEE_COL = None
APOGEE_COL = None

if celestrak is not None and PERIGEE_COL and APOGEE_COL:
    perigee = pd.to_numeric(celestrak[PERIGEE_COL], errors="coerce")
    apogee = pd.to_numeric(celestrak[APOGEE_COL], errors="coerce")
    mean_altitude = (perigee + apogee) / 2

    print(mean_altitude.describe())
    print("Share with mean altitude below 2,000 km:",
          round((mean_altitude < 2000).mean() * 100, 2), "%")

    mean_altitude.dropna().clip(upper=3000).hist(bins=60)
    plt.xlabel("Mean altitude (km)")
    plt.ylabel("Objects")
    plt.title("Preliminary altitude distribution")
    plt.show()
else:
    print("Set PERIGEE_COL and APOGEE_COL after schema inspection.")


## 9. Feasibility checklist

Before moving on to cleaning, answer these questions explicitly.

| Question | Status | Notes |
|---|---|---|
| Do we have a stable object identifier? | ⬜ | |
| Can UCS and CelesTrak be joined reliably? | ⬜ | |
| Can active satellites be distinguished from inactive objects? | ⬜ | |
| Can debris-related objects be identified consistently? | ⬜ | |
| Are launch dates sufficiently complete? | ⬜ | |
| Are decay dates sufficiently complete for historical in-orbit reconstruction? | ⬜ | |
| Do we have usable altitude data? | ⬜ | |
| Can we define LEO membership consistently? | ⬜ | |
| Is operator / country information usable for actor comparisons? | ⬜ | |
| Are there source-definition conflicts that require explicit harmonization? | ⬜ | |

**Do not proceed to a polished visualization until the critical items above have been resolved.**


## 10. Next step

After this audit, the next notebook/script should construct the first canonical `objects_master` dataset.

That step should convert every important methodological decision into reproducible code rather than manual spreadsheet edits.
